# ALPHA AI: past exams PDF to JSON
Run the cells in order. Cells 1 to 4 are safe (nothing is uploaded or deleted). Upload and delete are separate cells at the end.

In [ ]:
!apt-get -qq install -y poppler-utils > /dev/null
!pip -q install huggingface_hub requests
!rm -rf alpha-assets && git clone -q --depth 1 https://github.com/Kinfemichael-Alpha/alpha-assets
import sys
sys.path.insert(0, 'alpha-assets/tools')
import exam_extract, ai_enrich
print('ready')

## 1. Settings
Add these Colab secrets (key icon on the left): `HF_TOKEN` (a token with write access) and `API_KEYS`, a JSON list such as
`[{"provider":"gemini","key":"..."},{"provider":"groq","key":"..."},{"provider":"openrouter","key":"..."}]`.
Supported providers: gemini, groq, openai, openrouter, mistral, together, huggingface.

In [ ]:
import json
from google.colab import userdata
DATASET_REPO = 'Kinfemichael/CHANGE_ME'
HF_TOKEN = userdata.get('HF_TOKEN')
KEYS = json.loads(userdata.get('API_KEYS'))
WORK = 'data'
print(len(KEYS), 'API keys loaded')

## 2. Download the PDFs

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', local_dir=WORK, allow_patterns=['*.pdf'], token=HF_TOKEN)
import glob
print(len(glob.glob(WORK + '/**/*.pdf', recursive=True)), 'PDF files')

## 3. Extract questions, answers and explanations
This needs no AI. Every exam that has problems is listed so you can see what to check.

In [ ]:
report = exam_extract.main(WORK)
bad = [r for r in report if not r[1].startswith('OK')]
print()
print(len(report), 'exams,', len(bad), 'need review')

## 4. AI step: topics, question groups (passages) and sub/superscripts
Keys are used one at a time. When a key or model hits its limit the next one is used. When all are busy it waits one minute and starts again. It is safe to stop and run again: finished exams are skipped.

In [ ]:
import glob, json
pool = ai_enrich.Pool(KEYS, pause=60)
for path in sorted(glob.glob(WORK + '/**/*.json', recursive=True)):
    if path.endswith('exams_index.json'):
        continue
    exam = json.load(open(path, encoding='utf-8'))
    if exam.get('enriched'):
        continue
    ai_enrich.enrich(pool, exam)
    exam['enriched'] = True
    json.dump(exam, open(path, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('saved', path)

## 5. Build the index

In [ ]:
import os
index = []
for path in sorted(glob.glob(WORK + '/**/*.json', recursive=True)):
    if path.endswith('exams_index.json'):
        continue
    e = json.load(open(path, encoding='utf-8'))
    topics = {}
    for q in e['questions']:
        topics[q.get('topic') or 'Other'] = topics.get(q.get('topic') or 'Other', 0) + 1
    index.append({'id': e['id'], 'path': os.path.relpath(path, WORK).replace(os.sep, '/'), 'grade': e['grade'], 'subject': e['subject'], 'year_ec': e['year_ec'], 'round': e['round'], 'count': e['count'], 'time_minutes': e['time_minutes'], 'topics': topics, 'issues': len(e['issues']), 'enriched': e['enriched']})
json.dump({'schema': 1, 'exams': index}, open(WORK + '/exams_index.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(len(index), 'exams in the index')

## 6. Upload the JSON files (PDFs are not touched)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_folder(folder_path=WORK, repo_id=DATASET_REPO, repo_type='dataset', allow_patterns=['*.json'], commit_message='Add exam JSON files')
print('uploaded')

## 7. Optional: remove PDFs that now have a clean JSON
Only PDFs of exams with zero issues are removed. Set `CONFIRM_DELETE = True` yourself to run it. Keep the PDFs until you have checked a few exams in the app.

In [ ]:
CONFIRM_DELETE = False
if CONFIRM_DELETE:
    api = HfApi(token=HF_TOKEN)
    removed = 0
    for path in glob.glob(WORK + '/**/*.json', recursive=True):
        if path.endswith('exams_index.json'):
            continue
        e = json.load(open(path, encoding='utf-8'))
        if e['issues']:
            continue
        base = path[:-5]
        rel = os.path.relpath(base, WORK).replace(os.sep, '/')
        for suffix in ('_Entrance-Exam_Question-Paper.pdf', '_Entrance-Exam_Answer-Key.pdf', '_Question-Paper.pdf', '_Answer-Key.pdf'):
            try:
                api.delete_file(rel + suffix, repo_id=DATASET_REPO, repo_type='dataset', commit_message='Remove PDF replaced by JSON')
                removed += 1
            except Exception:
                pass
    print(removed, 'PDFs removed')
else:
    print('Deletion is off.')